In [ ]:
df_gptest2 = df[['body-style','price']]
grouped_test_bodystyle = df_gptest2.groupby(['body-style'], as_index=False).mean()
grouped_test_bodystyle

import matplotlib.pyplot as plt
%matplotlib inline 

# pcolor（pseudo-color，伪彩色图）把 DataFrame 当作一个数值矩阵，每个单元格画成一个色块，颜色深浅由数值大小决定；
#这里矩阵是 3 行（drive-wheels）× 5 列（body-style），单元格里存的是平均价格；cmap='RdBu' 指定红蓝配色：数值越大越红、越小越蓝——所以贵的组合呈红色，便宜或缺失（填了 0）的组合呈深蓝色。
# plt.pcolor(...) 函数式，默认 0、1、2 序号，不可读，差，画完就结束
plt.pcolor(grouped_pivot, cmap='RdBu')
# 在图右侧加一条颜色图例，把"颜色 ↔ 价格数值"的对应关系标出来，没有它图上的红蓝就无法量化。
plt.colorbar()
# 渲染并显示图形。在 Jupyter 中由于前面已有 %matplotlib inline，不加它通常也能显示，但这是标准的收尾写法。
plt.show()


# 同时创建两个对象并解包：
# fig（Figure）：整张画布，是所有元素的顶层容器； ax（Axes）：画布上的一块坐标系/绘图区，之后所有绘图操作都作用在它上面。 subplots() 不带参数时就是"1 张画布 + 1 个坐标系"。
fig, ax = plt.subplots()
# 和上一段的 plt.pcolor 内容相同：把透视表画成红蓝热力图（红=高价，蓝=低价/缺失填 0）；区别在于：调用者是 ax 而不是 plt，图画在指定的坐标系里；
# 返回值被存进了 im——它是一个 QuadMesh 对象（所有色块的集合）。这个对象非常重要，最后一行 fig.colorbar(im) 靠它才知道颜色条要按什么数值范围来标刻度。
im = ax.pcolor(grouped_pivot, cmap='RdBu')


#先看一下grouped_pivot的结构
#               price                                    ← 一级列（columns 第 0 层）
# body-style convertible  hardtop  hatchback ...         ← 二级列（columns 第 1 层）
# drive-wheels
# 4wd                 0.0       0.0  10311.76 ...
# fwd                 0.0   11894.5   9409.00 ...
# rwd           23633.00   22143.9  10363.44 ...

#grouped_pivot.columns，取出表头（即列名），这里标准叫法叫多级列索引（MultiIndex）
#有两层
# levels[0] → ['price']（未指定 values 参数，剩余数值列自动套上去的）
# levels[1] → ['convertible', 'hardtop', 'hatchback', 'sedan', 'wagon']（body-style 的取值）
row_labels = grouped_pivot.columns.levels[1]
# 行索引：grouped_pivot.index → Index(['4wd', 'fwd', 'rwd'], name='drive-wheels')
col_labels = grouped_pivot.index

# 这两行的作用是把刻度线移到每个色块的正中央，为下一行贴标签做准备。
#shape[0] 是行（取行数）、shape[1] 是列（取列数），顺序和数学里矩阵"m 行 n 列"的写法一致。grouped_pivot是3行5列
# grouped_pivot.shape[1]	5	透视表有 5 列（5 种 body-style）
#np.arange(5)	[0, 1, 2, 3, 4]	生成 0~4 的整数序列
#np.arange 是 NumPy 里的一个函数，用来生成等差数组（即按固定步长排列的一串数字）。步长默认为1
#np.arange(grouped_pivot.shape[1]) + 0.5 代表这个等差数组，所有元素右移半格，grouped_pivot.shape[1]这个值是5，即np.arange(5)；np.arange(1, 10, 2) # array([1, 3, 5, 7, 9])   步长为2
#minor=False 是什么意思？matplotlib 的刻度分两种：major（主刻度）：默认显示的大刻度；minor（次刻度）：更细的小刻度。
# minor=False 明确表示"我设置的是主刻度"。这里写出来主要是因为下一行 set_xticklabels(..., minor=False) 也要用同样参数配对——告诉 matplotlib"这些标签对应主刻度"，防止和次刻度混淆。
# ax.set_xticks(...)	—	把 X 轴刻度设在这些位置上；ax.set_yticks()同理
ax.set_xticks(np.arange(grouped_pivot.shape[1]) + 0.5, minor=False)
ax.set_yticks(np.arange(grouped_pivot.shape[0]) + 0.5, minor=False)


# 把 X 轴各刻度的显示文字依次设为 row_labels（body-style 名称）；
# 因为上一步 set_xticks 设了 5 个位置，所以按顺序对应
# 位置:  0.5      1.5      2.5     3.5    4.5
# 文字:  convertible  hardtop  hatchback  sedan  wagon
# 从此 X 轴下方显示的不再是 0 1 2 3 4，而是 5 种车身样式的名字。
#Y轴也一样
# 位置:  0.5    1.5    2.5
# 文字:  4wd    fwd    rwd
# set_xticks 和 set_xticklabels 必须成对使用，且数量一致
ax.set_xticklabels(row_labels, minor=False)
ax.set_yticklabels(col_labels, minor=False)

# rotation=90 把 X 轴刻度标签旋转 90 度，即竖着显示；
# 原因如注释所说：convertible、hatchback 这些单词较长，如果横着排，5 个标签会互相重叠成一团；竖排后每个标签只占一列的宽度，清晰可读；
# 注意这里用的是 plt.（pyplot 函数式），和前面 ax.set_xticklabels(...) 混用了两种接口——效果相同，pyplot 的 plt.xticks 会作用于当前活跃的坐标系（就是 ax）。
# 如果想保持纯 OOP 风格，等价写法是 ax.tick_params(axis='x', rotation=90) 或 ax.set_xticklabels(row_labels, rotation=90)。
plt.xticks(rotation=90)

# 在图右侧加一条竖直颜色条，标出“颜色 ↔ 数值”的对应关系（价格从低到高 = 从蓝到红）；
# 关键在参数 im：它是前面 ax.pcolor(...) 的返回值（QuadMesh 对象）。colorbar 靠它才知道：
# 这张图的颜色映射范围是什么（最小值 0 到最大值约 23633）；
# 用的是哪种 cmap（RdBu）；
# 这就是为什么当初画图时要把返回值存进 im——如果直接写 fig.colorbar()（旧版写法靠“最近一张图"的隐式状态），在多图场景下容易挂错图例。传 im 是明确、稳妥的写法。
fig.colorbar(im)  #fig是之前定义的画布对象，在画布中添加图例
# 触发渲染，把 fig 画布真正显示出来；
# 在 Jupyter 中因为之前有 %matplotlib inline，单元格执行完本会自动显示图，plt.show() 严格说可省，但它是脚本环境的通用标准写法，写上无害。
# 什么时候用 plt，什么时候用 ax 简单图随手用 plt（快速预览、一次性图表、不想改细节），需要精细控制或画多图时用 ax（要精细调整图的部件——本例就是典型：要挪刻度位置、换标签文字、转角度。）。
plt.show()

df.corr()

# SciPy（读作"sigh-pie"）是建立在 NumPy 之上的科学计算库，stats 是它的统计分析子模块，包含几百种统计检验、分布和函数；
# 写法是 from 库 import 子模块——因为 SciPy 很大，只把需要的 stats 拿进来，之后直接用 stats.xxx 调用，而不用写完整的 scipy.stats.xxx。
from scipy import stats

# stats.pearsonr(x, y) 一次返回两个值：
# 返回值	                         含义	                      例子（wheel-base vs price）
# pearson_coef	皮尔逊相关系数，范围 [-1, 1]，衡量线性关系强弱和方向	≈ 0.585
# p_value	p 值，衡量这个相关性是否统计显著（是不是碰巧出现的）	< 0.001  没有办法一口气算表格中所有的皮尔逊卡方统计量，只能算一对，但是能返回p值
pearson_coef, p_value = stats.pearsonr(df['wheel-base'], df['price'])
print("The Pearson Correlation Coefficient is", pearson_coef, " with a P-value of P =", p_value)  

# 方向：正相关
# r>0 → 轴距越长，车价总体越高。散点图上拟合线是向上倾斜的。

# 教材按 |r| 的常见划分：
# |r| 区间	线性关系
# 0 ~ 0.2	几乎没有
# 0.2 ~ 0.4	弱
# 0.4 ~ 0.6	中等 ← 0.585 在这里
# 0.6 ~ 0.8	中强
# 0.8 ~ 1.0	强

#                               df.corr()	                                                  stats.pearsonr(x, y)
# 范围	一次算出所有数值列两两组合的完整相关矩阵（第 92 行 df.corr() 一次给出二十多列的矩阵）	    一次只算一对变量
# p 值	❌ 不给 p 值，只有系数——你无法判断 0.585 是真相关还是运气	                        ✅ 同时返回 p 值，能做显著性判断
# 返回类型	DataFrame（对称矩阵，对角线恒为 1）	                                             元组 (coef, p_value)，可直接解包
# 一句话总结：两者算的是同一个皮尔逊系数，df.corr() 是“批量全景工具”（矩阵、无 p 值），stats.pearsonr 是“单对深挖工具”（系数 + 显著性检验）——教材先用前者扫出候选变量，再用后者逐一验证可信度，是标准的 EDA 相关性分析流程
# p值的作用（大样本和小样本的区别），这像“抓牌作弊”的推理——如果一个人连续 10 把摸到好牌（p 值极小），你敢说“纯属运气”吗？运气解释不通时，只剩一个结论：牌技有问题（真实关系存在）。
pearson_coef, p_value = stats.pearsonr(df['horsepower'], df['price'])
print("The Pearson Correlation Coefficient is", pearson_coef, " with a P-value of P = ", p_value)  

pearson_coef, p_value = stats.pearsonr(df['length'], df['price'])
print("The Pearson Correlation Coefficient is", pearson_coef, " with a P-value of P = ", p_value)  

pearson_coef, p_value = stats.pearsonr(df['width'], df['price'])
print("The Pearson Correlation Coefficient is", pearson_coef, " with a P-value of P =", p_value ) 

pearson_coef, p_value = stats.pearsonr(df['curb-weight'], df['price'])
print( "The Pearson Correlation Coefficient is", pearson_coef, " with a P-value of P = ", p_value)  


# 算出 r = 0.585
#       │
#       ▼
# 显著性检验（p 值）
#       │
#    ┌──┴───┐
#    ▼      ▼
# p 很小    p 很大
#    │      │
# 真相关    可能只是运气
# → 入选     → 淘汰，别当特征用